## Part 7: Targeted FPE Expanison

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import sys

class _DL:
 def __init__(self, f):
  self.t=sys.stdout; self.l=open(f, 'w')
 def write(self, m):
  self.t.write(m); self.l.write(m); self.l.flush()
 def flush(self):
  self.t.flush(); self.l.flush()
sys.stdout = _DL('live_output_part7.log')

# ======================================================================
# QUANTIZATION OPS (STE Base)
# ======================================================================
def quantize_w8(w):
    scale = 127.0 / w.abs().max().clamp(min=1e-8)
    w_q = torch.round(w * scale).clamp(-128, 127) / scale
    return w + (w_q - w).detach()

def quantize_w4(w):
    scale = 7.0 / w.abs().max().clamp(min=1e-8)
    w_q = torch.round(w * scale).clamp(-8, 7) / scale
    return w + (w_q - w).detach()

def quantize_w2(w):
    scale = w.abs().mean().clamp(min=1e-8)
    w_q = torch.round(w / scale).clamp(-1, 1) * scale
    return w + (w_q - w).detach()

def compute_projection_matrix(A, k):
    if A.shape[0] > 1000:
        _, _, V = torch.svd_lowrank(A, q=k)
        U_k = V
    else:
        L, V = torch.linalg.eigh(A)
        idx = torch.argsort(L, descending=True)
        V = V[:, idx]
        U_k = V[:, :k]
    
    P = U_k @ U_k.T
    return P

def analyze_phase_decay(alignments, start_idx, end_idx):
    slc = alignments[start_idx:end_idx]
    if len(slc) < 2:
        return 0, 0
        
    y = np.log(np.maximum(np.array(slc), 1e-12))
    x = np.arange(len(y)).reshape(-1, 1)
    
    reg = LinearRegression().fit(x, y)
    w = reg.coef_[0]
    r_sq = reg.score(x, y)
    return w, r_sq

def compute_fisher_pr(grads):
    B = grads.shape[0]
    F = (grads.T @ grads) / B
    trace_F = torch.trace(F)
    trace_F_sq = torch.trace(F @ F)
    if trace_F_sq < 1e-12: return 1.0 
    pr = (trace_F ** 2) / trace_F_sq
    return pr.item()

def compute_expansion_score(grads):
    B = grads.shape[0]
    g = grads.mean(dim=0)
    F = (grads.T @ grads) / B
    F_inv = torch.linalg.pinv(F, rcond=1e-5)
    eta = g.T @ F_inv @ g
    return eta.item(), torch.diag(F)

class ToyMatrixModel(nn.Module):
    def __init__(self, d_in, d_hidden, quant_type='W8A16'):
        super().__init__()
        self.d_hidden = d_hidden
        self.quant_type = quant_type
        self.W_in = nn.Parameter(torch.randn(d_in, d_hidden) / np.sqrt(d_in))
        self.W_out = nn.Parameter(torch.randn(d_hidden, d_in) / np.sqrt(d_hidden))
        
    def _quantize(self, w):
        if self.quant_type == 'W8A16': return quantize_w8(w)
        if self.quant_type == 'W4A8': return quantize_w4(w)
        if self.quant_type == 'Ternary': return quantize_w2(w)
        return w
        
    def forward(self, x):
        W_in_sim = self._quantize(self.W_in)
        W_out_sim = self._quantize(self.W_out)
        h = F.relu(x @ W_in_sim)
        return h @ W_out_sim

# ======================================================================
# EXPERIMENT 4: Targeted FPE Escape Verification (Part 7)
# Combines Deng Spectral Alignment, D_PR, and Eta thresholds simultaneously!
# ======================================================================
def run_exp4_targeted_escape(device, target_k=5):
    print("\n--- EXPERIMENT 4 (PART 7): TARGETED NEURON EXPANSION VIA COMBINED METRICS ---")
    d_in, d_hidden = 256, 64
    n_steps = 5000
    patience = 15
    tolerance = 0.05
    tau = 0.05
    
    model = ToyMatrixModel(d_in, d_hidden, quant_type='W4A8').to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.15, momentum=0.9)
    
    X = torch.randn(1024, d_in, device=device)
    Y = X @ (torch.randn(d_in, d_in, device=device)*0.5) 
    
    alignments = []
    pr_history = []
    plateau_counter = 0
    fpe_step = None
    
    print(f"Monitoring network geometry. Waiting for multi-metric superposition trap...")
    
    for step in range(n_steps):
        idx = torch.randint(0, 1024, (256,))
        x_b, y_b = X[idx], Y[idx]
        
        y_hat = model(x_b)
        loss = F.mse_loss(y_hat, y_b)
        
        optimizer.zero_grad()
        loss.backward()
        
        # 1. Deng Alignment Calculation
        g_mat = model.W_in.grad.detach() 
        A = g_mat @ g_mat.T 
        P = compute_projection_matrix(A, target_k)
        
        num = torch.norm(P @ g_mat)**2
        den = torch.norm(g_mat)**2 + 1e-12
        al_t = (num / den).item()
        alignments.append(al_t)
        
        # 2. SENN PR and Eta calculation
        pr_f = compute_fisher_pr(g_mat.T)
        eta, F_diag = compute_expansion_score(g_mat.T)
        
        pr_history.append(pr_f)
        if len(pr_history) > patience: pr_history.pop(0)
        
        if len(pr_history) == patience:
            d_pr = (pr_history[-1] - pr_history[0]) / patience
            if abs(d_pr) < tolerance:
                plateau_counter += 1
            else:
                plateau_counter = 0
        else:
            d_pr = float('inf')
            plateau_counter = 0

        # Trigger check if we haven't expanded yet
        
        if step % 100 == 0:
            print(f"  [Step {step:4d}] Loss: {loss.item():.4f} | AL_t: {al_t:.3f} | D_PR: {pr_f:.2f} (d={d_pr:.4f}) | plateau={plateau_counter}/{patience} | dEta: {delta_eta:.4f} / req: {req_delta:.4f}")

        if fpe_step is None and plateau_counter >= patience:
            # Check Eta vs Rho boundary
            threshold_var = torch.median(F_diag)
            prop_variance = F_diag[F_diag >= threshold_var].sum() / F_diag.sum()
            delta_eta = eta * prop_variance.item()
            req_delta = tau * (eta / pr_f)
            
            # Require both D_PR plateau AND Eta boundary AND high Deng alignment trap
            if delta_eta >= req_delta and al_t > 0.72:
                fpe_step = step
                print(f"\n[Step {step}] 🎯 MULTI-METRIC TRAP DETECTED:")
                print(f"  - Deng Alignment (AL_t): {al_t:.3f} (>0.72 threshold)")
                print(f"  - Fisher D_PR Plateaud: {d_pr:.4f} < {tolerance}")
                print(f"  - Natural Expansion Delta Eta: {delta_eta:.4f} >= {req_delta:.4f}")
                
                print(f"  --> Executing TARGETED fractional expansion!")
                
                to_expand = (F_diag >= threshold_var).nonzero(as_tuple=True)[0]
                n_added = len(to_expand)
                print(f"  --> Expanding exclusively {n_added} saturated neurons (out of {d_hidden}).")
                
                old_w_in = model.W_in.detach()
                old_w_out = model.W_out.detach()
                
                spawn_w_in = torch.randn(d_in, n_added, device=device) * 0.01
                spawn_w_out = torch.randn(n_added, d_in, device=device) * 0.01
                
                model.W_in = nn.Parameter(torch.cat([old_w_in, spawn_w_in], dim=1))
                model.W_out = nn.Parameter(torch.cat([old_w_out, spawn_w_out], dim=0))
                model.d_hidden += n_added
                
                optimizer = torch.optim.SGD(model.parameters(), lr=0.08, momentum=0.9)
                
        optimizer.step()
    
    if fpe_step:
        decay_w, decay_r2 = analyze_phase_decay(alignments, fpe_step+10, fpe_step+110)
        print(f"\nDecay Rate after Targeted FPE Injection: w = {decay_w:.4f} (R^2 = {decay_r2:.2f})")
        if decay_w < 0:
            print("✅ SUCCESS: Targeted mathematical escape phase (negative slope) established using minimal capacity guided by joint metrics.")
        else:
            print("❌ FAILURE: Network collapsed back into alignment trap. Targeted expansion was too weak.")
    else:
        print("❌ FAILURE: Network never reached a true multi-metric plateau state to trigger expansion.")
        
    return alignments, fpe_step
    
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run_exp4_targeted_escape(device)

main()




